# 04 — SFT Training

Fine-tune GPT-2 on a tiny instruction dataset using `SFTTrainer`.

This notebook is intentionally small (10 examples, 2 epochs) so it runs in ~1 minute on Colab CPU.

> **Colab users:** Run the setup cell, restart runtime, then continue.

In [ ]:
import os, sys, subprocess

def _is_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def _install():
    IN_COLAB = _is_colab()
    if IN_COLAB:
        repo = 'MyLLM'
        if not os.path.exists(repo):
            print('Cloning repository...')
            r = subprocess.run(
                ['git', 'clone', 'https://github.com/silvaxxx1/MyLLM.git', repo],
                capture_output=True, text=True
            )
            if r.returncode != 0:
                print('Clone failed:\n', r.stderr); return
        print('Installing myllm...')
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', f'./{repo}'],
            capture_output=True, text=True
        )
    else:
        root = os.path.abspath(os.path.join(os.getcwd(), '..'))
        print(f'Installing from {root} ...')
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-e', root],
            capture_output=True, text=True
        )
    if r.returncode != 0:
        print('Install FAILED. Error output:')
        print(r.stdout[-3000:] if r.stdout else '')
        print(r.stderr[-3000:] if r.stderr else '')
    else:
        msg = 'Restart runtime now (Runtime → Restart runtime).' if IN_COLAB else 'Ready.'
        print('Done!', msg)

try:
    import myllm
    print(f'myllm {myllm.__version__} already installed.')
except ImportError:
    _install()

## 1. Imports

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer as HFTokenizer

from myllm import ModelConfig
from myllm.train import SFTTrainer, SFTTrainerConfig

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch :', torch.__version__)
print('device:', device)

## 2. Tiny instruction dataset

In [ ]:
EXAMPLES = [
    {'instruction': 'What is the capital of France?',      'response': 'The capital of France is Paris.'},
    {'instruction': 'What is 2 + 2?',                     'response': '2 + 2 equals 4.'},
    {'instruction': 'Name a planet in our solar system.',  'response': 'Earth is a planet in our solar system.'},
    {'instruction': 'What color is the sky?',              'response': 'The sky is blue during the day.'},
    {'instruction': 'What is machine learning?',           'response': 'Machine learning is a subfield of AI where models learn from data.'},
    {'instruction': 'Who wrote Romeo and Juliet?',         'response': 'Romeo and Juliet was written by William Shakespeare.'},
    {'instruction': 'What is the speed of light?',         'response': 'The speed of light is approximately 299,792 km/s.'},
    {'instruction': 'What does DNA stand for?',            'response': 'DNA stands for Deoxyribonucleic Acid.'},
    {'instruction': 'What is a neural network?',           'response': 'A neural network is a computational model inspired by the human brain.'},
    {'instruction': 'What year did World War II end?',     'response': 'World War II ended in 1945.'},
]

TEMPLATE = '### Instruction:\n{instruction}\n\n### Response:\n{response}'

print('Sample formatted example:')
print(TEMPLATE.format(**EXAMPLES[0]))

## 3. PyTorch Dataset

In [ ]:
class InstructionDataset(Dataset):
    def __init__(self, examples, tokenizer, max_length=128):
        self.items = []
        for ex in examples:
            enc = tokenizer(
                TEMPLATE.format(**ex),
                max_length=max_length,
                padding='max_length',
                truncation=True,
                return_tensors='pt',
            )
            ids = enc['input_ids'].squeeze(0)
            self.items.append({
                'input_ids':      ids,
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels':         ids.clone(),
            })

    def __len__(self):          return len(self.items)
    def __getitem__(self, i):   return self.items[i]


tokenizer = HFTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

dataset    = InstructionDataset(EXAMPLES, tokenizer)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

print(f'Dataset : {len(dataset)} examples  |  {len(dataloader)} batches/epoch')
print(f'Shape   : {next(iter(dataloader))["input_ids"].shape}')

## 4. Configure the trainer

In [ ]:
trainer_cfg = SFTTrainerConfig(
    output_dir='./demo_sft_output',
    num_epochs=2,
    batch_size=2,
    gradient_accumulation_steps=1,
    max_grad_norm=1.0,
    logging_steps=1,
    save_steps=0,       # no checkpoints in this demo
    report_to=[],       # no WandB
    model_config_name='gpt2-small',
)

print('SFTTrainerConfig ready.')
print(f'  output_dir = {trainer_cfg.output_dir}')
print(f'  num_epochs = {trainer_cfg.num_epochs}')

## 5. Train

In [ ]:
model_cfg = ModelConfig.from_name('gpt2-small')

trainer = SFTTrainer(trainer_cfg, model_config=model_cfg)
trainer.setup_model()
trainer.setup_data(train_dataloader=dataloader)
trainer.setup_optimizer()

trainer.train()

## 6. Generate from the fine-tuned model

In [ ]:
from myllm import LLM, GenerationConfig

# Wrap trained model in LLM for generation
llm = LLM(config=model_cfg, device=str(trainer.device))
llm.model = trainer.model
llm.model.eval()

gen_cfg = GenerationConfig(
    max_length=40, temperature=0.7,
    top_k=50, apply_top_k_sampling=True,
    do_sample=True, use_kv_cache=True,
    use_optimized_sampler=True,
)

test_prompts = [
    '### Instruction:\nWhat is the capital of France?\n\n### Response:\n',
    '### Instruction:\nWhat is 2 + 2?\n\n### Response:\n',
]

for prompt in test_prompts:
    result = llm.generate_text(prompt, tokenizer, gen_cfg)
    print(result['text'])
    print('-' * 60)

## 7. Save the fine-tuned weights

In [ ]:
import os

save_path = './demo_sft_output/finetuned_gpt2.pt'
llm.save(save_path)
print('Saved to:', os.path.abspath(save_path))